## Lecture 8: Computer Arithmetic & Numerical Accuracy

### ****Exercise 1:** Machine Epsilon**

****Goal:** find machine epsilon experimentally by successive halving, then compare to the standard value `np.finfo(dtype).eps`.**

In [1]:
# From slide 31

import numpy as np

def find_machine_epsilon(dtype=np.float64):
    eps = dtype(1.0)
    while dtype(1.0) + eps / dtype(2.0) != dtype(1.0):
        eps = eps / dtype(2.0)
    return eps

for dtype in [np.float16, np.float32, np.float64]:
    computed = find_machine_epsilon(dtype)
    reference = np.finfo(dtype).eps
    print(f"{dtype.__name__}:")
    print(f" Computed: {float(computed):.4e}")
    print(f" np.finfo: {float(reference):.4e}")
    print()

float16:
 Computed: 9.7656e-04
 np.finfo: 9.7656e-04

float32:
 Computed: 1.1921e-07
 np.finfo: 1.1921e-07

float64:
 Computed: 2.2204e-16
 np.finfo: 2.2204e-16



****Algorithm:****
1. **Start with `eps = dtype(1.0)`**
2. **Repeat: `eps = eps / 2 until dtype(1.0) + eps/2 == dtype(1.0)`**
3. **The last value where `1.0 + eps/2 != 1.0` is machine epsilon**

Done! Please see the code above.

****Run for:** `np.float16`, `np.float32`, `np.float64`**

Done. Please see the output above.

****Done?** Do your computed values match `np.finfo`? → discuss with a neighbour.**

Yes they do match.

### ****Exercise 2:** Catastrophic Cancellation**

****Goal:** observe precision loss when subtracting nearly-equal numbers, and fix it by reformulating the algorithm.**

****Test polynomial: $x^2 − 10000.0001x + 1 = 0$****
- **Roots: $x_1 \approx 10000.0001$ and $x_2 \approx 10^{−4}$**
- **In the naive formula, computing $x_2$ requires subtracting two large, nearly-equal numbers — losing all significant digits in float32**

In [2]:
# From slide 33 (naive formula)

import numpy as np

def quadratic_naive(a, b, c):
    t = type(a)                        # np.float32 or np.float64
    disc = t(np.sqrt(b*b - t(4)*a*c)) # b*b not b**2; t() casts literals and sqrt
    x1 = (-b + disc) / (t(2)*a)
    x2 = (-b - disc) / (t(2)*a)
    return x1, x2
# x^2 - 10000.0001*x + 1 = 0 roots: x1 ~ 10000, x2 ~ 1e-4
for dtype in [np.float32, np.float64]:
    a, b, c = dtype(1.0), dtype(-10000.0001), dtype(1.0)
    x1, x2 = quadratic_naive(a, b, c)
    print(f"{dtype.__name__}: x1 = {float(x1):.4f}, x2 = {float(x2):.10f}")

float32: x1 = 10000.0000, x2 = 0.0000000000
float64: x1 = 10000.0000, x2 = 0.0001000000


In [3]:
# From slide 34 (stable formula)
# I added np.float16 as well

import numpy as np

def quadratic_stable(a, b, c):
    t = type(a)
    disc = t(np.sqrt(b*b - t(4)*a*c))
    if b > 0:
        x1 = (-b - disc) / (t(2)*a) # pick sign that avoids cancellation
    else:
        x1 = (-b + disc) / (t(2)*a)
    x2 = c / (a * x1) # Vieta’s formula: x1 * x2 = c/a
    return x1, x2

true_small = 1.0 / 10000.0001    # ~ 1e-4
for dtype in [np.float16, np.float32, np.float64]: 
    a, b, c = dtype(1.0), dtype(-10000.0001), dtype(1.0)
    _, x2_naive = quadratic_naive(a, b, c)
    _, x2_stable = quadratic_stable(a, b, c)
    err_naive = abs(float(x2_naive) - true_small) / true_small
    err_stable = abs(float(x2_stable) - true_small) / true_small
    print(f"{dtype.__name__}: naive={err_naive:.2e} stable={err_stable:.2e}")

float16: naive=inf stable=1.00e+00
float32: naive=1.00e+00 stable=1.53e-08
float64: naive=1.20e-08 stable=1.00e-08


/tmp/ipykernel_1433/1093832580.py:7: RuntimeWarning: overflow encountered in scalar multiply
  disc = t(np.sqrt(b*b - t(4)*a*c)) # b*b not b**2; t() casts literals and sqrt
/tmp/ipykernel_1433/570804700.py:8: RuntimeWarning: overflow encountered in scalar multiply
  disc = t(np.sqrt(b*b - t(4)*a*c))


****Tasks:****

1. **Implement the naive quadratic formula; run for float32 and float64**

Done! Please see the code above.

2. **Implement the stable version using Vieta’s formula $(x_1 \cdot x_2 = c/a)$**

Done! Please see the code above.

3. **Compare relative error for the small root $x_2$ in both versions**

Done! Please see the code above.

****NumPy dtype tip:** In NumPy < 2.0, Python int literals, `b**2`, and `np.sqrt` silently upcast float32 to float64. Inside your function add `t = type(a)` and use `b*b, t(4)*a*c, t(2)*a, disc = t(np.sqrt(...))` to keep everything in the target dtype.**

Noted!

****Done?** Record the relative error (naive vs stable, float32) → discuss with a neighbour.**

In the case of naive float16, we see a overflow which was expected. The naive float32 is also wrong. Only the naive float64 seems to get it right as well as the stable methods.

### ****Exercise 3 (Optional):** Error Accumulation**

****Task:** add `0.1` in a loop *n* times; compare the result against the exact value *n × 0.1* for float32 and float64.**

In [4]:
# From slide 36

import numpy as np

n_values = [10, 100, 1_000, 10_000, 100_000]

for dtype in [np.float32, np.float64]:
    print(f"\n{dtype.__name__}:")
    for n in n_values:
        total = dtype(0.0)
        for _ in range(n):
            total += dtype(0.1)
        expected = n * 0.1
        rel_error = abs(float(total) - expected) / expected
        print(f" n={n:>7d}: result={float(total):.10f} rel_error={rel_error:.2e}")


float32:
 n=     10: result=1.0000001192 rel_error=1.19e-07
 n=    100: result=10.0000019073 rel_error=1.91e-07
 n=   1000: result=99.9990463257 rel_error=9.54e-06
 n=  10000: result=999.9028930664 rel_error=9.71e-05
 n= 100000: result=9998.5566406250 rel_error=1.44e-04

float64:
 n=     10: result=1.0000000000 rel_error=1.11e-16
 n=    100: result=10.0000000000 rel_error=1.95e-15
 n=   1000: result=100.0000000000 rel_error=1.41e-14
 n=  10000: result=1000.0000000002 rel_error=1.59e-13
 n= 100000: result=10000.0000000188 rel_error=1.88e-12


- **Try $n \in \{10, 100, 1000, 10000, 100000\}$**

Done! Please see the code above.

- **Compute relative error: `|result - expected| / expected`**

Done! Please see the code above.

- **Does the error grow with *n*? Is float64 better?**

Yes the error grows with n. Float64 is better with lower errors.

**Why does 0.1 cause problems? It has no exact binary representation — like 1/3 in decimal, it repeats forever. Each addition compounds the truncation error.**

In [5]:
print(np.binary_repr(np.float32(0.1).view(np.int32), width=32))
print(np.binary_repr(np.float64(0.1).view(np.int64), width=64))

00111101110011001100110011001101
0011111110111001100110011001100110011001100110011001100110011010


****Optional reading:** Eijkhout §3.6.2 (Summing series) discusses related accumulation patterns and summation order strategies.**

Skipped.

****Done?** Observe how error grows with *n* → discuss with a neighbour**

The error becomes larger and larger as *n* grows.